# Day 3 Project - Image Map & Similarity Search

**Path B (creative project).** Turn a collection of images into vectors with a pretrained **CNN (ResNet18)**, then:
1. draw a **2D UMAP map** of the collection (the story-telling artifact), and
2. build a **cosine similarity search** - "show me images most like this one".

Runs locally on Apple GPU (MPS). CIFAR-10 is already downloaded in `./data`, but you can point it at **your own folder of images** instead (see the marked cell).

> Minimal working pipeline. Extend it: swap in your own photos, add fine-tuning, or build the optional Streamlit app.

## 1. Setup

In [ ]:
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")  # let unsupported ops fall back to CPU on Mac

import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

def pick_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

device = pick_device()
print("device:", device)

N_IMAGES = 600  # how many images to embed; bump it up for a richer map

## 2. Load images

Using a slice of **CIFAR-10** (already downloaded in `./data`). To use **your own images**, comment out this cell and use the next one.

In [ ]:
from torchvision import datasets

cifar = datasets.CIFAR10(root="./data", train=False, download=True)  # file is already here, so this just verifies + extracts
classes = cifar.classes

img_pils   = [cifar[i][0] for i in range(N_IMAGES)]
img_labels = [classes[cifar[i][1]] for i in range(N_IMAGES)]
print("loaded", len(img_pils), "images |", len(set(img_labels)), "classes:", sorted(set(img_labels)))

In [ ]:
# --- OPTIONAL: use your OWN folder of images instead of CIFAR ---
# Put images in a folder, then uncomment. Labels come from subfolder names
# (folder/cats/*.jpg, folder/dogs/*.jpg). If no subfolders, labels are all "mine".
#
# from PIL import Image
# import glob
# paths = sorted(glob.glob("my_images/**/*.jpg", recursive=True))[:N_IMAGES]
# img_pils   = [Image.open(p).convert("RGB") for p in paths]
# img_labels = [os.path.basename(os.path.dirname(p)) or "mine" for p in paths]
# print("loaded", len(img_pils), "of my own images")

## 3. Images to vectors (ResNet18)

Run each image through a CNN pretrained on ImageNet and read the 512-dim vector just before the classifier - a compact **image2vec**.

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.models.feature_extraction import create_feature_extractor

weights = ResNet18_Weights.IMAGENET1K_V1
preprocess = weights.transforms()  # matching resize + normalize
cnn = resnet18(weights=weights).eval().to(device)
extractor = create_feature_extractor(cnn, return_nodes={"flatten": "feat"})

def image_vectors(pils, batch=32):
    "embed a list of PIL images into a (N, 512) numpy array"
    out = []
    for i in tqdm(range(0, len(pils), batch), desc="image2vec"):
        x = torch.stack([preprocess(p) for p in pils[i:i+batch]]).to(device)
        with torch.no_grad():
            out.append(extractor(x)["feat"].float().cpu().numpy())
    return np.concatenate(out, axis=0)

img_vectors = image_vectors(img_pils)
print("embedded", img_vectors.shape[0], "images into", img_vectors.shape[1], "dims")

## 4. The artifact: a 2D UMAP map

UMAP squeezes the 512-dim vectors down to 2D so we can *see* the collection. Points that land near each other looked similar to the CNN. Colored by class - if classes form clean islands, the embedding is meaningful.

In [ ]:
import umap  # package is umap-learn

reducer = umap.UMAP(n_components=2, random_state=42)  # first call is slow (numba compiles)
coords = reducer.fit_transform(img_vectors)

plt.figure(figsize=(9, 7))
labels_arr = np.array(img_labels)
for c in sorted(set(img_labels)):
    pts = coords[labels_arr == c]
    plt.scatter(pts[:, 0], pts[:, 1], s=16, label=c)
plt.legend(markerscale=2, fontsize=8, loc="best")
plt.title(f"UMAP map of {len(img_pils)} images (ResNet18 embeddings)")
plt.tight_layout()
plt.savefig("umap_map.png", dpi=140)  # saved artifact
plt.show()

## 5. Similarity search

Given one image, return the most similar others by **cosine similarity** of their vectors.

In [ ]:
def cosine_neighbors(idx, vectors, topn=6):
    "indices + scores of the topn images most similar to image `idx`"
    m = vectors / (np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-9)
    sims = m @ m[idx]
    order = np.argsort(-sims)
    order = order[order != idx][:topn]  # drop the query itself
    return order, sims[order]

query = 0  # try different indices
neighbors, scores = cosine_neighbors(query, img_vectors, topn=5)

fig, ax = plt.subplots(1, 6, figsize=(14, 3))
ax[0].imshow(img_pils[query]); ax[0].set_title(f"query\n{img_labels[query]}"); ax[0].axis("off")
for j, (n, s) in enumerate(zip(neighbors, scores), start=1):
    ax[j].imshow(img_pils[n]); ax[j].set_title(f"{img_labels[n]}\n{s:.2f}"); ax[j].axis("off")
plt.suptitle("nearest neighbors by cosine similarity")
plt.tight_layout(); plt.show()

## 6. Save your vectors

In [ ]:
np.save("embeddings.npy", img_vectors)
np.save("labels.npy", np.array(img_labels, dtype=object))
print("saved embeddings.npy, labels.npy, umap_map.png")

## Where to take it next (your turn)

- **Make it yours:** point cell 2 at your own folder of images (pets, plants, screenshots, memes) - that satisfies the brief's "data you gathered yourself".
- **Fine-tune:** unfreeze `layer4` and train a small head on your own labels (see Tool 3 in the toolbox).
- **Sharper story:** annotate a few points on the UMAP map, or find the pair of images the CNN thinks are most/least alike.
- **Optional app:** the toolbox writes a small `streamlit` search app from `embeddings.npy` / `labels.npy`.